# Bert Model

In [ ]:
pip install -q transformers datasets accelerate evaluate

## Imports

In [ ]:
import torch
import transformers
import datasets

print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

In [ ]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Data

In [ ]:
x_train = joblib.load("/kaggle/input/datasets/sudheermuthyala8688/data12/x_train.pkl")
x_test = joblib.load("/kaggle/input/datasets/sudheermuthyala8688/data12/x_test.pkl")

y_train = joblib.load("/kaggle/input/datasets/sudheermuthyala8688/data12/y_train.pkl")
y_test = joblib.load("/kaggle/input/datasets/sudheermuthyala8688/data12/y_test.pkl")

In [ ]:
print(type(x_train))
print(type(y_train))

print(len(x_train))
print(len(x_test))

print(x_train[:3])
print(y_train[:3])

Convert to Hugging Face Dataset

In [ ]:
tr_df = pd.DataFrame(
    {
        "text" : x_train,
        "label": y_train
    }
)

te_df = pd.DataFrame(
    {
        "text" : x_test,
        "label": y_test
    }
)
print(tr_df.head())

In [ ]:
train_data = Dataset.from_pandas(tr_df)
test_data  = Dataset.from_pandas(te_df)
print(train_data)
print(test_data)

For your first BERT experiment, don't train on all 1.27 million tweets. It will take several hours on a T4 GPU.

In [ ]:
train_df = tr_df.sample(
    n=100000,
    random_state=42
).reset_index(drop=True)

test_df = te_df.sample(
    n=25000,
    random_state=42
).reset_index(drop=True)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Tokenizer

In [ ]:
MODEL_NAME = "bert-base-uncased"

In [ ]:
bert_tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
MAX_LENGTH = 64

def tokenize_function(batch):
    return bert_tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
print(bert_tokenizer)
print(type(bert_tokenizer))

In [ ]:
train_dataset = train_dataset.map(
      tokenize_function,
      batched=True,
      batch_size=1000
  )

test_dataset = test_dataset.map(
      tokenize_function,
      batched=True,
      batch_size=1000
  )

In [ ]:
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

if "__index_level_0__" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["__index_level_0__"])
    test_dataset = test_dataset.remove_columns(["__index_level_0__"])

Rename Label Column

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

Convert to PyTorch Format

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [ ]:
print(train_dataset)
print(train_dataset[0])

## Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Training Arguments

In [ ]:
train_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=True,
    report_to="none"
)

Data Collator

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer
)

Metrics

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)

  precision, recall, f1, _ = precision_recall_fscore_support(
      labels,
      predictions,
      average="binary"
  )
  accuracy = accuracy_score(labels, predictions)

  return {
      "accuracy": accuracy,
      "precision": precision,
      "recall": recall,
      "f1": f1
  }

Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=bert_tokenizer
)

In [ ]:
import torch
import torchvision
import datasets
import transformers

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)

In [ ]:
import time
start_time = time.time()
trainer.train()
end_time = time.time()

print(f"Training Time: {(end_time - start_time) / 60:.2f} minutes")

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
import json

evaluation_results = {
    "Evaluation Loss": 0.8950701355934143,
    "Accuracy": 0.79936,
    "Precision": 0.8124226676565207,
    "Recall": 0.7822253990945913,
    "F1 Score": 0.7970381160475843,
    "Evaluation Runtime": 64.2416,
    "Samples per Second": 389.156,
    "Steps per Second": 6.086
}

with open("bert_evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("Evaluation results saved successfully.")

In [ ]:
trainer.save_model("bert_sentiment_model")
bert_tokenizer.save_pretrained("bert_sentiment_model")

print("BERT model saved successfully!")

In [ ]:
import shutil

shutil.make_archive("bert_sentiment_model", "zip", "bert_sentiment_model")

print("ZIP file created successfully!")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot(cmap="Blues")
plt.title("BERT Confusion Matrix")
plt.show()

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="bert_sentiment_model",
    tokenizer="bert_sentiment_model"
)

texts = [
    "I love this movie!",
    "This is the worst experience ever.",
    "The service was average."
]

for text in texts:
    result = classifier(text)
    print(text)
    print(result)
    print("-" * 50)